# Battery lifetime pipeline — Colab runner

Disk-light workflow: nothing is downloaded to your laptop. The notebook
1. mounts your Drive (or fetches data from the public Drive folders),
2. clones the latest GitHub repo into Colab,
3. runs the full v2 pipeline (audit → features → splits → VIF report → experiments) inside Colab,
4. zips the generated CSVs / JSON results, and
5. downloads the ZIP to your laptop so you can `git add` it and push.

Re-run anytime by clicking **Runtime → Run all**.

## 0. Configuration

In [ ]:
# Edit these to match your setup if needed.

GITHUB_REPO = "https://github.com/osmansafacifci/Graduation-Project-Dicle.git"
BRANCH = "main"

# Where the raw data lives in your Drive (used if MOUNT_DRIVE=True).
MATR_DRIVE_DIR = "/content/drive/MyDrive/Braatz_NatEnergy2019"   # contains batch1.pkl, batch2.pkl, batch3.pkl
HUST_DRIVE_DIR = "/content/drive/MyDrive/HUST"                  # contains 1-1.pkl, 1-2.pkl, ..., 10-8.pkl

# If True: mount Drive and symlink the folders above into the repo.
# If False: use gdown to fetch from the public shared folders into the repo's data/raw/.
MOUNT_DRIVE = True

# Pipeline stages to run. Leave as None to run all.
# e.g. STAGES_TO_RUN = ['audit_matr', 'audit_hust', 'features']
STAGES_TO_RUN = None

# Optional toggles (forwarded into the v2 scripts).
EOL_FRACTION = 0.85          # SOP 0.80 → bumped to 0.85 by supervisor for batch1/3 to be modelable
EXTRA_WINDOWS = []           # add 25 here if you want N=25 ablation (default windows are [50, 100])

# Model lineup. Default is all 6 models (ElasticNet, PLS, Random Forest, XGBoost,
# CatBoost, Gaussian Process). Override by listing a subset, e.g.
#   EXTRA_MODELS = ['pls', 'xgboost']
EXTRA_MODELS = []

CAPACITY_NORMALIZE = False   # SOP §2.3: only flip when adding a third dataset with a different cell
VIF_DROP = False             # default report-only; set True to actually prune VIF>5 features and run a separate ablation

## 1. Clone the repo and install dependencies

In [ ]:
import os, subprocess, shutil
from pathlib import Path

REPO_DIR = Path('/content/Graduation-Project-Dicle')
if REPO_DIR.exists():
    print('[clone] repo already present, pulling latest')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    print(f'[clone] {GITHUB_REPO} -> {REPO_DIR}')
    subprocess.run(['git', 'clone', '--branch', BRANCH, GITHUB_REPO, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
# Install the lightweight set the v2 pipeline actually uses.
# (The repo's full requirements.txt is heavy because it pins jupyter/etc; on
#  Colab those are already there.)
%pip install -q gdown h5py xgboost catboost scikit-learn pandas numpy scipy

## 2. Make raw data available under data/raw/

Two paths — `MOUNT_DRIVE = True` symlinks your already-downloaded MyDrive folders into
the repo (no copy, no extra disk). Otherwise we use gdown against the public shared
folders (Anyone-with-link, Viewer).

In [ ]:
RAW_DIR = REPO_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
HUST_LINK = RAW_DIR / 'HUST_data'

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    matr_src = Path(MATR_DRIVE_DIR)
    hust_src = Path(HUST_DRIVE_DIR)
    assert matr_src.exists(), f'MATR Drive folder not found: {matr_src}'
    assert hust_src.exists(), f'HUST Drive folder not found: {hust_src}'
    # Symlink the three MATR pkls into data/raw/
    for fname in ('batch1.pkl', 'batch2.pkl', 'batch3.pkl'):
        target = matr_src / fname
        link = RAW_DIR / fname
        if link.exists() or link.is_symlink():
            link.unlink()
        if not target.exists():
            print(f'[warn] {target} missing in Drive — skipping')
            continue
        link.symlink_to(target)
        print(f'[link] {link.name}  ->  {target}')
    # Symlink the HUST folder
    if HUST_LINK.exists() or HUST_LINK.is_symlink():
        if HUST_LINK.is_symlink():
            HUST_LINK.unlink()
        else:
            shutil.rmtree(HUST_LINK)
    HUST_LINK.symlink_to(hust_src)
    print(f'[link] {HUST_LINK}  ->  {hust_src}')
else:
    print('[gdown] fetching from public shared folders (~15-20 GB into Colab disk)')
    subprocess.run(['python', '0_data_prep/download_data.py'], check=True)

# Sanity check
for p in [RAW_DIR / 'batch1.pkl', RAW_DIR / 'batch2.pkl', RAW_DIR / 'batch3.pkl', HUST_LINK]:
    print('  exists' if p.exists() else '  MISSING', p)

## 3. Run the pipeline

In [ ]:
# Stage 1: audits (runs straight from Drive-linked data, no extra config needed)
stages = STAGES_TO_RUN or ['audit_matr', 'audit_hust']
for stage in ['audit_matr', 'audit_hust']:
    if stage in stages:
        subprocess.run(['python', 'run_pipeline.py', '--skip-download', '--stages', stage], check=True)

In [ ]:
# Stage 2: features — pass through optional knobs
if STAGES_TO_RUN is None or 'features' in STAGES_TO_RUN:
    windows = sorted(set([50, 100, *EXTRA_WINDOWS]))
    cmd = [
        'python', '1_feature_engineering/build_sop12_features_v2.py',
        '--eol-fraction', str(EOL_FRACTION),
        '--n-windows', *[str(w) for w in windows],
    ]
    if CAPACITY_NORMALIZE:
        cmd.append('--capacity-normalize')
    print('$', ' '.join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# Stage 3: splits
if STAGES_TO_RUN is None or 'splits' in STAGES_TO_RUN:
    subprocess.run(['python', '2_modeling_featuring/generate_sop_splits_v2.py'], check=True)

In [ ]:
# Stage 4: VIF (report-only by default; flip VIF_DROP=True to prune)
if STAGES_TO_RUN is None or 'vif' in STAGES_TO_RUN:
    cmd = ['python', '2_modeling_featuring/vif_screening.py']
    if VIF_DROP:
        cmd.append('--drop')
    print('$', ' '.join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# Stage 5: experiments
# Default (always runs): all 6 models on all 12 features -> outputs/results_v2/
#   models = ElasticNet, PLS, Random Forest, XGBoost, CatBoost, Gaussian Process
# When VIF_DROP=True, ALSO runs the VIF-pruned ablation -> outputs/results_v2_vif_drop/
ALL_MODELS = ['elastic_net', 'pls', 'random_forest', 'xgboost', 'catboost', 'gaussian_process']

if STAGES_TO_RUN is None or 'experiments' in STAGES_TO_RUN:
    windows = sorted(set([50, 100, *EXTRA_WINDOWS]))
    models = list(EXTRA_MODELS) if EXTRA_MODELS else ALL_MODELS

    # baseline run (all 12 SOP features)
    cmd = [
        'python', '2_modeling_featuring/run_experiments_v2.py',
        '--windows', *[str(w) for w in windows],
        '--models', *models,
    ]
    print('$', ' '.join(cmd))
    subprocess.run(cmd, check=True)

    # VIF-pruned ablation (requires VIF stage to have run with --drop and produced vif_kept_features.txt)
    if VIF_DROP:
        kept_path = REPO_DIR / 'data' / 'intermediate' / 'vif_kept_features.txt'
        if not kept_path.exists():
            print('[warn] VIF_DROP=True but vif_kept_features.txt missing — make sure VIF stage ran with --drop.')
        else:
            ablation_cmd = [
                'python', '2_modeling_featuring/run_experiments_v2.py',
                '--windows', *[str(w) for w in windows],
                '--models', *models,
                '--features-from', str(kept_path),
                '--output-dir', str(REPO_DIR / 'outputs' / 'results_v2_vif_drop'),
            ]
            print('$', ' '.join(ablation_cmd))
            subprocess.run(ablation_cmd, check=True)

## 4. Pack outputs into a ZIP and download

We bundle the small CSV/JSON artifacts (no raw .pkl) so you can unzip locally
and `git add data/intermediate splits/sop_v2 outputs/results_v2`.

In [ ]:
import datetime, zipfile

stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
zip_path = Path('/content') / f'pipeline_outputs_{stamp}.zip'

INCLUDE_DIRS = [
    REPO_DIR / 'data' / 'intermediate',
    REPO_DIR / 'splits' / 'sop_v2',
    REPO_DIR / 'outputs' / 'results_v2',
    REPO_DIR / 'outputs' / 'results_v2_vif_drop',  # only present if VIF_DROP=True
]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for d in INCLUDE_DIRS:
        if not d.exists():
            print(f'[skip] {d} (does not exist)')
            continue
        for path in d.rglob('*'):
            if path.is_file():
                arcname = path.relative_to(REPO_DIR)
                zf.write(path, arcname)
                print(f'  +{arcname}')

print(f'\n[zip] wrote {zip_path}  ({zip_path.stat().st_size / (1024*1024):.2f} MB)')

In [ ]:
# Trigger the browser download.
from google.colab import files
files.download(str(zip_path))

## 5. Local steps after download

On your laptop:

```bash
cd /Users/osmancifci/Graduation-Project-Dicle
unzip -o ~/Downloads/pipeline_outputs_*.zip
git add data/intermediate splits/sop_v2 outputs/results_v2
# if you ran with VIF_DROP=True, also include the ablation dir:
git add outputs/results_v2_vif_drop
git commit -m 'pipeline run from Colab (<date>)'
git push
```

These are small CSV/JSON artifacts (a few MB total), so they fit comfortably in Git.

## VIF drop ablation — separate run

To produce the VIF-pruned ablation results in addition to the baseline:

1. In cell 0, set:
   ```python
   VIF_DROP = True
   ```
2. **Runtime → Run all** again. This will:
   - re-run VIF screening with `--drop` (writes `vif_kept_features.txt`),
   - run experiments twice: once with all 12 features (`outputs/results_v2/`)
     and once with the VIF-pruned subset (`outputs/results_v2_vif_drop/`).
3. Download the new ZIP, unzip, and commit the additional `outputs/results_v2_vif_drop/`
   directory alongside the baseline.